# Phase 3D — Local Manual 30-Run Notebook (v3)

Run the setup cell first. It must print **SETUP COMPLETE (v3 JSON-safe local runner)**. Then run the 30 experiment cells one-by-one and finally the aggregation cell.

v3 fixes MOFA+/NumPy metadata JSON serialization and preserves any partial failed experiment directory by moving it into `08_experiments/_partial_failed_phase3d/` before rerun.


In [1]:
# SETUP — run once. This cell MUST end with "SETUP COMPLETE".

from pathlib import Path
import os, sys, json, hashlib, subprocess, platform, time
from importlib.metadata import version

REPO_ROOT = Path("/Users/rifatahasan/Documents/ChatGPT/multiomics-research").resolve()
RUNNER_SOURCE = REPO_ROOT / "code" / "benchmarks" / "run_phase3d.py"
CONFIG_DIR = REPO_ROOT / "07_models" / "02_classical_integration" / "configs"
LOCAL_DIR = REPO_ROOT / ".local_manual_phase3d"
TEMP_CONFIG_DIR = LOCAL_DIR / "configs"
LOG_DIR = LOCAL_DIR / "logs"
LOCAL_RUNNER = LOCAL_DIR / "run_phase3d_local.py"

TEMP_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Python:", platform.python_version())
print("Machine:", platform.platform())

# -------------------------------------------------------------------
# 1. Hard environment check: do not silently change the frozen env.
# -------------------------------------------------------------------
expected_versions = {
    "numpy": "2.4.4",
    "scipy": "1.17.1",
    "scikit-learn": "1.9.1",
    "h5py": "3.16.0",
    "mofapy2": "0.7.5",
    "POT": "0.9.6.post1",
}

if platform.python_version() != "3.11.8":
    raise RuntimeError(
        f"Use the Phase 3D Python 3.11 kernel. Expected Python 3.11.8, got {platform.python_version()}."
    )

for package, expected in expected_versions.items():
    actual = version(package)
    print(f"{package}: {actual}")
    if actual != expected:
        raise RuntimeError(f"{package}: expected {expected}, got {actual}")

# -------------------------------------------------------------------
# 2. Verify official SCOT source.
# -------------------------------------------------------------------
SCOT_ROOT = REPO_ROOT / "external" / "SCOT"
SCOT_COMMIT = "14649be6e14017dcfe7ba619091b33d1df55f6a9"

if not (SCOT_ROOT / ".git").exists():
    SCOT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/rsinghlab/SCOT.git", str(SCOT_ROOT)],
        check=True
    )

subprocess.run(["git", "-C", str(SCOT_ROOT), "fetch", "--all", "--tags"], check=True)
subprocess.run(
    ["git", "-C", str(SCOT_ROOT), "checkout", "--detach", SCOT_COMMIT],
    check=True
)
actual_scot = subprocess.check_output(
    ["git", "-C", str(SCOT_ROOT), "rev-parse", "HEAD"], text=True
).strip()
if actual_scot != SCOT_COMMIT:
    raise RuntimeError(f"SCOT commit mismatch: {actual_scot}")

print("SCOT source verified:", actual_scot)

# -------------------------------------------------------------------
# 3. Repository state.
# -------------------------------------------------------------------
CURRENT_HEAD = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
).strip()

WORKTREE_STATUS = subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=REPO_ROOT, text=True
)
WORKTREE_STATUS_SHA256 = hashlib.sha256(WORKTREE_STATUS.encode()).hexdigest()

print("Research Git HEAD:", CURRENT_HEAD)
print("Research working tree clean:", not bool(WORKTREE_STATUS.strip()))
if WORKTREE_STATUS.strip():
    print("Working-tree status SHA256:", WORKTREE_STATUS_SHA256)

# -------------------------------------------------------------------
# 4. Verify exactly the 30 expected frozen configs.
# -------------------------------------------------------------------
EXPECTED_CONFIGS = [
    "EXP-LN-A1-MOFAPLUS-KMEANS-S1729.json",
    "EXP-LN-A1-MOFAPLUS-KMEANS-S2718.json",
    "EXP-LN-A1-MOFAPLUS-KMEANS-S31415.json",
    "EXP-LN-A1-SCOT-KMEANS-S1729.json",
    "EXP-LN-A1-SCOT-KMEANS-S2718.json",
    "EXP-LN-A1-SCOT-KMEANS-S31415.json",

    "EXP-LN-D1-MOFAPLUS-KMEANS-S1729.json",
    "EXP-LN-D1-MOFAPLUS-KMEANS-S2718.json",
    "EXP-LN-D1-MOFAPLUS-KMEANS-S31415.json",
    "EXP-LN-D1-SCOT-KMEANS-S1729.json",
    "EXP-LN-D1-SCOT-KMEANS-S2718.json",
    "EXP-LN-D1-SCOT-KMEANS-S31415.json",

    "EXP-MB-E11-MOFAPLUS-KMEANS-S1729.json",
    "EXP-MB-E11-MOFAPLUS-KMEANS-S2718.json",
    "EXP-MB-E11-MOFAPLUS-KMEANS-S31415.json",
    "EXP-MB-E11-SCOT-KMEANS-S1729.json",
    "EXP-MB-E11-SCOT-KMEANS-S2718.json",
    "EXP-MB-E11-SCOT-KMEANS-S31415.json",

    "EXP-MB-E13-MOFAPLUS-KMEANS-S1729.json",
    "EXP-MB-E13-MOFAPLUS-KMEANS-S2718.json",
    "EXP-MB-E13-MOFAPLUS-KMEANS-S31415.json",
    "EXP-MB-E13-SCOT-KMEANS-S1729.json",
    "EXP-MB-E13-SCOT-KMEANS-S2718.json",
    "EXP-MB-E13-SCOT-KMEANS-S31415.json",

    "EXP-MB-E15-MOFAPLUS-KMEANS-S1729.json",
    "EXP-MB-E15-MOFAPLUS-KMEANS-S2718.json",
    "EXP-MB-E15-MOFAPLUS-KMEANS-S31415.json",
    "EXP-MB-E15-SCOT-KMEANS-S1729.json",
    "EXP-MB-E15-SCOT-KMEANS-S2718.json",
    "EXP-MB-E15-SCOT-KMEANS-S31415.json",
]

missing = [name for name in EXPECTED_CONFIGS if not (CONFIG_DIR / name).exists()]
if missing:
    raise FileNotFoundError("Missing frozen configs:\n" + "\n".join(missing))

print("Verified 30 frozen configs.")

# -------------------------------------------------------------------
# 5. Build a LOCAL copy of the repository runner.
#
# We do not import run_phase3d.py into Jupyter because IPython/Jupyter may
# already have Python's stdlib module named `code` loaded, which conflicts
# with the repository package `code.benchmarks`.
#
# Scientific logic remains copied directly from the current repository
# runner. Only execution-environment gates/root resolution are amended.
# -------------------------------------------------------------------

source = RUNNER_SOURCE.read_text(encoding="utf-8")

# Fix local JSON serialization for scientific metadata returned by libraries.
# MOFA+ can return numpy arrays/scalars inside metadata (for example variance
# explained structures). The repository's basic JSON writer uses stdlib
# json.dumps directly, so the local runner wraps it with a deterministic
# numpy-safe conversion without changing any scientific values.
old_io_import = "from code.benchmarks.common.io import write_json_atomic"
new_io_import = "from code.benchmarks.common.io import write_json_atomic as _base_write_json_atomic"
if old_io_import not in source:
    raise RuntimeError("Could not locate write_json_atomic import in run_phase3d.py")
source = source.replace(old_io_import, new_io_import, 1)

serialization_patch = r"""
def _json_ready(value):
    import numpy as _np
    if isinstance(value, _np.ndarray):
        return value.tolist()
    if isinstance(value, _np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(v) for v in value]
    return value

def write_json_atomic(value, path):
    return _base_write_json_atomic(_json_ready(value), path)

"""
# Insert the wrapper immediately after imports and before ROOT.
root_marker = 'ROOT = Path(__file__).resolve().parents[2]'
if root_marker not in source:
    raise RuntimeError("Could not locate ROOT definition in run_phase3d.py")
source = source.replace(root_marker, serialization_patch + root_marker, 1)

old_root = 'ROOT = Path(__file__).resolve().parents[2]'
new_root = 'ROOT = Path(os.environ["PHASE3D_REPO_ROOT"]).resolve()'
if old_root not in source:
    raise RuntimeError("Could not locate ROOT definition in run_phase3d.py")
source = source.replace(old_root, new_root, 1)

old_allowed = 'ALLOWED_COMPUTE = {"COLAB_CPU", "COLAB_HIGH_MEMORY"}'
new_allowed = 'ALLOWED_COMPUTE = {"COLAB_CPU", "COLAB_HIGH_MEMORY", "LOCAL_MANUAL"}'
if old_allowed not in source:
    raise RuntimeError("Could not locate ALLOWED_COMPUTE definition in run_phase3d.py")
source = source.replace(old_allowed, new_allowed, 1)

# Existing source has a Colab-controller gate. For LOCAL_MANUAL we retain a
# positive explicit execution gate, but rename the user-facing error semantics.
old_gate = 'if os.environ.get("ASTRA_COLAB_CONFIRMED") != "1":\n        raise RuntimeError("set ASTRA_COLAB_CONFIRMED=1 only inside the verified Colab controller")'
new_gate = 'if os.environ.get("ASTRA_PHASE3D_EXECUTION_CONFIRMED") != "1":\n        raise RuntimeError("set ASTRA_PHASE3D_EXECUTION_CONFIRMED=1 only from the verified Phase 3D controller")'
if old_gate not in source:
    raise RuntimeError("Could not locate execution confirmation gate in run_phase3d.py")
source = source.replace(old_gate, new_gate, 1)

LOCAL_RUNNER.write_text(source, encoding="utf-8")
print("Local runner prepared:", LOCAL_RUNNER)

# -------------------------------------------------------------------
# 6. Prepare temporary configs with only execution/provenance changes.
# -------------------------------------------------------------------
def prepare_local_config(config_name):
    src = CONFIG_DIR / config_name
    original = json.loads(src.read_text(encoding="utf-8"))

    cfg = dict(original)
    cfg["start_commit"] = CURRENT_HEAD
    cfg["compute_classification"] = "LOCAL_MANUAL"
    cfg["manual_execution"] = {
        "mode": "MANUAL_LOCAL_NOTEBOOK",
        "source_config": str(src.relative_to(REPO_ROOT)),
        "source_start_commit": original.get("start_commit"),
        "local_git_head": CURRENT_HEAD,
        "working_tree_clean": not bool(WORKTREE_STATUS.strip()),
        "working_tree_status_sha256": WORKTREE_STATUS_SHA256,
    }

    temp = TEMP_CONFIG_DIR / config_name
    temp.write_text(json.dumps(cfg, indent=2) + "\n", encoding="utf-8")
    return temp, original

# -------------------------------------------------------------------
# 7. Execute ONE experiment in a fresh Python subprocess.
# -------------------------------------------------------------------
def run_experiment(config_name):
    if config_name not in EXPECTED_CONFIGS:
        raise ValueError(f"Not a frozen Phase 3D config: {config_name}")

    temp_config, original = prepare_local_config(config_name)
    experiment_id = original["experiment_id"]
    output_dir = REPO_ROOT / "08_experiments" / experiment_id

    print("=" * 100)
    print("EXPERIMENT:", experiment_id)
    print("DATASET:", original["dataset"])
    print("METHOD:", original["method"])
    print("SEED:", original["seed"])
    print("=" * 100)

    if output_dir.exists():
        status_path = output_dir / "STATUS"
        status = status_path.read_text().strip() if status_path.exists() else "INCOMPLETE"
        if status == "SUCCEEDED":
            print("Already SUCCEEDED — skipping.")
            return

        # Preserve any partial failed output for audit instead of deleting it.
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        quarantine_root = REPO_ROOT / "08_experiments" / "_partial_failed_phase3d"
        quarantine_root.mkdir(parents=True, exist_ok=True)
        quarantine = quarantine_root / f"{experiment_id}__{status}__{timestamp}"
        output_dir.rename(quarantine)
        print(f"Previous partial output preserved at: {quarantine}")

    env = os.environ.copy()
    env["PHASE3D_REPO_ROOT"] = str(REPO_ROOT)
    env["ASTRA_PHASE3D_EXECUTION_CONFIRMED"] = "1"
    env["PYTHONPATH"] = str(REPO_ROOT)
    env["OMP_NUM_THREADS"] = "1"
    env["OPENBLAS_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"
    env["VECLIB_MAXIMUM_THREADS"] = "1"
    env["LOKY_MAX_CPU_COUNT"] = "1"

    log_base = LOG_DIR / experiment_id
    stdout_path = log_base.with_suffix(".stdout.log")
    stderr_path = log_base.with_suffix(".stderr.log")

    command = [
        sys.executable,
        str(LOCAL_RUNNER),
        "--config",
        str(temp_config),
    ]

    print("Launching local process...")
    t0 = time.time()
    result = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    elapsed = time.time() - t0

    stdout_path.write_text(result.stdout or "", encoding="utf-8")
    stderr_path.write_text(result.stderr or "", encoding="utf-8")

    if result.stdout:
        print("\n--- stdout ---")
        print(result.stdout)

    if result.stderr:
        print("\n--- stderr ---")
        print(result.stderr)

    print(f"\nReturn code: {result.returncode}")
    print(f"Elapsed: {elapsed:.2f} seconds")
    print("stdout log:", stdout_path)
    print("stderr log:", stderr_path)

    if result.returncode != 0:
        raise RuntimeError(f"{experiment_id} failed with return code {result.returncode}")

    print("\nSUCCEEDED:", experiment_id)

print("\nSETUP COMPLETE (v3 JSON-safe local runner) — run experiment cells one-by-one.")
print("run_experiment defined:", callable(run_experiment))

Repository: /Users/rifatahasan/Documents/ChatGPT/multiomics-research
Python: 3.11.8
Machine: macOS-26.6.2-arm64-arm-64bit
numpy: 2.4.4
scipy: 1.17.1
scikit-learn: 1.9.1
h5py: 3.16.0
mofapy2: 0.7.5
POT: 0.9.6.post1
SCOT source verified: 14649be6e14017dcfe7ba619091b33d1df55f6a9
Research Git HEAD: c7732e2b916086f17122610f5ce2be991d62e259
Research working tree clean: False
Working-tree status SHA256: d24eb4fe6cf00892a254d5ed14b5bf71b525a942a42b3e18bb7302c9e93eb110
Verified 30 frozen configs.
Local runner prepared: /Users/rifatahasan/Documents/ChatGPT/multiomics-research/.local_manual_phase3d/run_phase3d_local.py

SETUP COMPLETE (v3 JSON-safe local runner) — run experiment cells one-by-one.
run_experiment defined: True


HEAD is now at 14649be Merge pull request #17 from rsinghlab/pinardemetci-patch-1


In [2]:
# Experiment 01/30
run_experiment("EXP-LN-A1-MOFAPLUS-KMEANS-S1729.json")


EXPERIMENT: EXP-LN-A1-MOFAPLUS-KMEANS-S1729
DATASET: LN_A1
METHOD: MOFAPLUS
SEED: 1729
Previous partial output preserved at: /Users/rifatahasan/Documents/ChatGPT/multiomics-research/08_experiments/_partial_failed_phase3d/EXP-LN-A1-MOFAPLUS-KMEANS-S1729__FAILED__20260912_141104
Launching local process...

--- stdout ---

        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Scaling views to unit variance...

Successfully loaded view='RNA' g

RuntimeError: EXP-LN-A1-MOFAPLUS-KMEANS-S1729 failed with return code 1

In [ ]:
# Experiment 02/30
run_experiment("EXP-LN-A1-MOFAPLUS-KMEANS-S2718.json")


In [ ]:
# Experiment 03/30
run_experiment("EXP-LN-A1-MOFAPLUS-KMEANS-S31415.json")


In [ ]:
# Experiment 04/30
run_experiment("EXP-LN-A1-SCOT-KMEANS-S1729.json")


In [ ]:
# Experiment 05/30
run_experiment("EXP-LN-A1-SCOT-KMEANS-S2718.json")


In [ ]:
# Experiment 06/30
run_experiment("EXP-LN-A1-SCOT-KMEANS-S31415.json")


In [ ]:
# Experiment 07/30
run_experiment("EXP-LN-D1-MOFAPLUS-KMEANS-S1729.json")


In [ ]:
# Experiment 08/30
run_experiment("EXP-LN-D1-MOFAPLUS-KMEANS-S2718.json")


In [ ]:
# Experiment 09/30
run_experiment("EXP-LN-D1-MOFAPLUS-KMEANS-S31415.json")


In [ ]:
# Experiment 10/30
run_experiment("EXP-LN-D1-SCOT-KMEANS-S1729.json")


In [ ]:
# Experiment 11/30
run_experiment("EXP-LN-D1-SCOT-KMEANS-S2718.json")


In [ ]:
# Experiment 12/30
run_experiment("EXP-LN-D1-SCOT-KMEANS-S31415.json")


In [ ]:
# Experiment 13/30
run_experiment("EXP-MB-E11-MOFAPLUS-KMEANS-S1729.json")


In [ ]:
# Experiment 14/30
run_experiment("EXP-MB-E11-MOFAPLUS-KMEANS-S2718.json")


In [ ]:
# Experiment 15/30
run_experiment("EXP-MB-E11-MOFAPLUS-KMEANS-S31415.json")


In [ ]:
# Experiment 16/30
run_experiment("EXP-MB-E11-SCOT-KMEANS-S1729.json")


In [ ]:
# Experiment 17/30
run_experiment("EXP-MB-E11-SCOT-KMEANS-S2718.json")


In [ ]:
# Experiment 18/30
run_experiment("EXP-MB-E11-SCOT-KMEANS-S31415.json")


In [ ]:
# Experiment 19/30
run_experiment("EXP-MB-E13-MOFAPLUS-KMEANS-S1729.json")


In [ ]:
# Experiment 20/30
run_experiment("EXP-MB-E13-MOFAPLUS-KMEANS-S2718.json")


In [ ]:
# Experiment 21/30
run_experiment("EXP-MB-E13-MOFAPLUS-KMEANS-S31415.json")


In [ ]:
# Experiment 22/30
run_experiment("EXP-MB-E13-SCOT-KMEANS-S1729.json")


In [ ]:
# Experiment 23/30
run_experiment("EXP-MB-E13-SCOT-KMEANS-S2718.json")


In [ ]:
# Experiment 24/30
run_experiment("EXP-MB-E13-SCOT-KMEANS-S31415.json")


In [ ]:
# Experiment 25/30
run_experiment("EXP-MB-E15-MOFAPLUS-KMEANS-S1729.json")


In [ ]:
# Experiment 26/30
run_experiment("EXP-MB-E15-MOFAPLUS-KMEANS-S2718.json")


In [ ]:
# Experiment 27/30
run_experiment("EXP-MB-E15-MOFAPLUS-KMEANS-S31415.json")


In [ ]:
# Experiment 28/30
run_experiment("EXP-MB-E15-SCOT-KMEANS-S1729.json")


In [ ]:
# Experiment 29/30
run_experiment("EXP-MB-E15-SCOT-KMEANS-S2718.json")


In [ ]:
# Experiment 30/30
run_experiment("EXP-MB-E15-SCOT-KMEANS-S31415.json")


In [ ]:
# FINAL CELL — collect all 30 outputs into one results table + aggregates.

import csv
import json
import math
from collections import defaultdict

EXP_ROOT = REPO_ROOT / "08_experiments"
RESULTS_PATH = EXP_ROOT / "PHASE_3D_MANUAL_LOCAL_RESULTS.csv"
AGG_PATH = EXP_ROOT / "PHASE_3D_MANUAL_LOCAL_AGGREGATES.csv"

def extract_metric(metrics, key):
    value = metrics.get(key)
    if isinstance(value, dict):
        return value.get("result")
    return value

rows = []

for config_name in EXPECTED_CONFIGS:
    cfg = json.loads((CONFIG_DIR / config_name).read_text(encoding="utf-8"))
    exp_id = cfg["experiment_id"]
    exp_dir = EXP_ROOT / exp_id

    status = "NOT_RUN"
    if (exp_dir / "STATUS").exists():
        status = (exp_dir / "STATUS").read_text().strip()

    metrics = {}
    validation = {}
    provenance = {}

    if (exp_dir / "metrics.json").exists():
        metrics = json.loads((exp_dir / "metrics.json").read_text())
    if (exp_dir / "validation.json").exists():
        validation = json.loads((exp_dir / "validation.json").read_text())
    if (exp_dir / "provenance.json").exists():
        provenance = json.loads((exp_dir / "provenance.json").read_text())

    rows.append({
        "experiment_id": exp_id,
        "dataset": cfg["dataset"],
        "method": cfg["method"],
        "seed": cfg["seed"],
        "ARI": extract_metric(metrics, "ARI"),
        "NMI": extract_metric(metrics, "NMI"),
        "silhouette": extract_metric(metrics, "silhouette"),
        "runtime_seconds": provenance.get("runtime_seconds"),
        "execution_status": status,
        "scientific_qc_status": validation.get("scientific_qc_status", ""),
        "experiment_directory": str(exp_dir),
    })

with RESULTS_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

def as_float(value):
    try:
        x = float(value)
        return x if math.isfinite(x) else None
    except Exception:
        return None

grouped = defaultdict(list)
for row in rows:
    if row["execution_status"] == "SUCCEEDED":
        grouped[(row["dataset"], row["method"])].append(row)

aggregate_rows = []
for (dataset, method), items in sorted(grouped.items()):
    record = {"dataset": dataset, "method": method, "n_runs": len(items)}
    for metric in ["ARI", "NMI", "silhouette", "runtime_seconds"]:
        vals = [as_float(x[metric]) for x in items]
        vals = [x for x in vals if x is not None]
        if vals:
            mean = sum(vals) / len(vals)
            if len(vals) > 1:
                sd = (sum((x - mean) ** 2 for x in vals) / (len(vals) - 1)) ** 0.5
            else:
                sd = 0.0
            record[f"{metric}_mean"] = mean
            record[f"{metric}_sd"] = sd
        else:
            record[f"{metric}_mean"] = ""
            record[f"{metric}_sd"] = ""
    aggregate_rows.append(record)

agg_fields = [
    "dataset", "method", "n_runs",
    "ARI_mean", "ARI_sd",
    "NMI_mean", "NMI_sd",
    "silhouette_mean", "silhouette_sd",
    "runtime_seconds_mean", "runtime_seconds_sd",
]

with AGG_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=agg_fields)
    writer.writeheader()
    writer.writerows(aggregate_rows)

print("Saved:", RESULTS_PATH)
print("Saved:", AGG_PATH)

print("\nIndividual runs:")
for row in rows:
    print(
        f"{row['experiment_id']:<48} "
        f"{row['execution_status']:<10} "
        f"ARI={row['ARI']} NMI={row['NMI']} silhouette={row['silhouette']}"
    )

print("\nAggregates:")
for row in aggregate_rows:
    print(row)

print("\nAccounting:")
print("SUCCEEDED:", sum(r["execution_status"] == "SUCCEEDED" for r in rows))
print("NOT_RUN:", sum(r["execution_status"] == "NOT_RUN" for r in rows))
print("TOTAL:", len(rows))